# Wave Equation Factorization via Pseudo-Differential Operators
## From Second-Order Hyperbolic to First-Order Schrödinger-type Evolution

This notebook demonstrates how to factorize the classical 2D wave equation $\partial_{tt} u = \Delta u$ into two first-order evolution equations using the pseudo-differential fractional power $(-\Delta)^{1/2}$. 

Unlike standard finite-difference approaches, using the `PseudoDifferentialOperator` framework allows us to:
1. Rigorously compute the exact symbol of the fractional Laplacian.
2. Automatically derive the **microlocal asymptotic corrections** if the wave speed becomes spatially dependent $c(x,y)$.
3. Extract the **Hamiltonian flow** (ray trajectories) directly from the principal symbol.

---

## 1. The Governing Equation & Factorization

The standard wave equation is:
$$
\partial_{tt} u - \Delta u = 0
$$

In Fourier space, $\Delta \to -(\xi^2 + \eta^2)$. We can factorize the differential operator algebraically:
$$
(\partial_t - \sqrt{\Delta})(\partial_t + \sqrt{\Delta}) u = 0
$$

Since $\Delta$ is negative-definite, its square root is purely imaginary: $\sqrt{\Delta} = i(-\Delta)^{1/2}$. 
Defining the pseudo-differential operator $P = (-\Delta)^{1/2}$ with symbol $p(\xi, \eta) = \sqrt{\xi^2 + \eta^2}$, the wave equation splits into two independent first-order equations:

$$
\partial_t u = \pm i P u \quad \Longleftrightarrow \quad \partial_t u = \pm i (-\Delta)^{1/2} u
$$

The $(+)$ sign corresponds to **forward-propagating** waves, and the $(-)$ sign to **backward-propagating** waves.

# Implementation
## 0. Imports 

In [ ]:
from solver import PDESolver, psiOp
from psiop import PseudoDifferentialOperator
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML 

## 1. Physical and simulation parameters 

In [ ]:
# ── Grid and Time ──
Lx, Ly   = 10.0, 10.0
Nx, Ny = 128, 128    

Lt, Nt   = 5.0, 200
n_frames = 100 

## 2. Grid setup and SymPy symbols 

In [ ]:
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')   # shape (Nx, Ny) 

t, x, y   = sp.symbols('t x y', real=True)
xi, eta   = sp.symbols('xi eta', real=True, positive=True)
psi_func  = sp.Function('psi')
psi_field = psi_func(t, x, y) 

## 3. Constructing the Fractional Laplacian

We first define the standard Laplacian $\Delta$ (symbol: $-(\xi^2 + \eta^2)$), and then use the new `fractional_power` method to compute $\sqrt{\Delta}$. 

*Note: For a constant coefficient operator, this yields exactly $i\sqrt{\xi^2+\eta^2}$. If we had defined a variable speed $c(x,y)$, this method would automatically compute the asymptotic spatial corrections!*

In [ ]:
# ── 1. Define the Laplacian ──
symbol_laplacian = -(xi**2 + eta**2)
Lap = PseudoDifferentialOperator(symbol_laplacian, [x, y], mode='symbol')

print("Principal symbol of Laplacian:")
print("  a(ξ, η) =", symbol_laplacian)

# ── 2. Compute the fractional power (Square Root) ──
# alpha=0.5 for square root. order=1 includes the first microlocal corrections 
# (which are zero here for constant coefficients, but the machinery is active).
sqrt_lap_symbol = Lap.fractional_power(alpha=0.5, order=1, method='symbolic')

print("\nSymbol of the fractional Laplacian √(Δ):")
print("  a_1/2(ξ, η) =", sqrt_lap_symbol)

# ── 3. Verify the order of the new operator ──
sqrt_lap_op = PseudoDifferentialOperator(sqrt_lap_symbol, [x, y], mode='symbol')
print("\nAsymptotic order of √(Δ):", sqrt_lap_op.symbol_order()) 

## 4. The Factored First-Order Equations

Using the computed symbol, we construct the forward-propagating wave equation:
$$
\partial_t\psi = i \Psi_{\text{op}}\!\left( \sqrt{\xi^2 + \eta^2} \right) \psi
$$

This is mathematically equivalent to a dispersive Schrödinger equation where the dispersion relation is linear ($\omega = |k|$) rather than quadratic ($\omega = |k|^2$).

In [ ]:
# ∂ψ/∂t = psiOp(√(ξ² + η²), ψ)
#        ─────────────────────────────
#        Forward propagating wave (Fractional Laplacian)

equation_forward = sp.Eq(
    sp.diff(psi_field, t),
    psiOp(sqrt_lap_symbol, psi_field)
)

print("Factored Forward Wave Equation:")
print("  ∂ψ/∂t = psiOp(√(ξ² + η²), ψ)") 

## 5. Initial conditions: A localized wave packet 

In [ ]:
def initial_condition_wave(xx, yy):
    """
    A localized Gaussian wave packet with a carrier frequency.
    """
    A0 = 1.0
    W0 = 1.0       # Beam waist
    X0 = -3.0      # Initial x-position
    Y0 = 0.0       # Initial y-position
    KX = 3.0       # Carrier frequency (controls group velocity)
    KY = 0.0
    
    env = np.exp(-((xx - X0)**2 + (yy - Y0)**2) / W0**2)
    phase = KX * xx + KY * yy
    
    return A0 * env * np.exp(1j * phase) 

## 6. Solver setup 

In [ ]:
solver = PDESolver(equation_forward)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='periodic',
    initial_condition=initial_condition_wave,
    n_frames=n_frames,
    plot=True,
) 

## 7. Solve 

In [ ]:
frames1 = solver.solve() 

## 8. Visualization of the Wave Evolution 

In [ ]:
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',   # Show the real part of the wave field
    overlay='contour',  
    mode='surface',
    physical=True
)

HTML(ani.to_jshtml()) 

In [ ]:
ani.save('wave_factorization_forward.mp4', writer='ffmpeg', fps=20, dpi=100)
print("✅ Saved to wave_factorization_forward.mp4") 

In [ ]:
    x, xi = sp.symbols('x xi', real=True)
    op = PseudoDifferentialOperator(xi**2 + x, [x], mode='symbol')
    
    # order=1 prevents the exp(alpha * log P) series from expanding into massive terms
    res_frac = op.fractional_power(alpha=0.5, order=1, method='symbolic')
    assert res_frac is not None
    from sympy import Expr
    assert isinstance(res_frac, Expr)

In [ ]:
op.symbol